# BattMo.jl Workshop — Setup & Hello World

**Agenda slot:** 9:30 – 10:30

If you haven't already worked through **`Installation_check.ipynb`**, do that first — it makes sure Julia, VSCode and the required packages are correctly installed. This notebook assumes that setup is done, and takes you through your first BattMo.jl simulations.

By the end of this hour you will:
- Understand how BattMo.jl structures its inputs (Parameters and Settings)
- Run your first battery simulation
- Explore and visualize the output
- Make your first parameter change (an assignment, to warm up for the rest of the day)

## Setup check

Let's import BattMo and the plotting packages we'll use throughout the day.

In [ ]:
using BattMo, GLMakie, Jutul

## 1 - Input ([docs](https://battmoteam.github.io/BattMo.jl/dev/manuals/user_guide/terminology))

BattMo.jl structures its simulation inputs into two primary categories: Parameters and Settings. This distinction helps users differentiate between the physical characteristics of the battery system and the numerical configurations of the simulation.

**Parameters** represent the controllable variables in real-world experiments. They are further divided into:
- **Cell Parameters**: define the intrinsic properties of the battery cell, such as geometry and material characteristics.
- **Cycling Protocol Parameters**: specify how the cell is operated during a simulation.

**Settings** are used to configure numerical assumptions for solving equations and finding numerical solutions. They are further divided into:
- **Model Settings**: define numerical assumptions related to the battery model, such as diffusion methods or simplifications used in the simulation.
- **Simulation Settings**: define numerical assumptions specific to the simulation.

BattMo stores cell parameters, cycling protocols and settings in a user-friendly JSON format to facilitate reuse. We can load parameters directly from the built-in default sets, which is very convenient for quickly testing a simulation setup. Let's see which default sets BattMo provides.

In [ ]:
print_default_input_sets()

For our example, we'll load the cell parameter set of an NMC811 vs. Graphite-SiOx cell whose parameters were determined in the [Chen 2020 paper](https://doi.org/10.1149/1945-7111/ab9050), together with a simple Constant Current Discharge cycling protocol.

In [ ]:
cell_parameters = load_cell_parameters(; from_default_set = "chen_2020")
cycling_protocol = load_cycling_protocol(; from_default_set = "cc_discharge")

A loaded cell parameter set is a Dictionary-like object which comes with some additional handy functions. First, let's list the outermost keys of the cell parameters object.

In [ ]:
keys(cell_parameters)

Now we access the `Separator` key.

In [ ]:
cell_parameters["Separator"]

We have a flat list of parameters and values for the separator. In other cases, a key might nest other dictionaries, which can be accessed using the normal dictionary notation. Let's look at the active material parameters of the negative electrode.

In [ ]:
cell_parameters["NegativeElectrode"]["ActiveMaterial"]

There are many parameters, nested into dictionaries. Often we are more interested in a specific subset of parameters. We can find a parameter with the `search_parameter` function. For example, let's see how area-related parameters are named:

In [ ]:
search_parameter(cell_parameters, "area")

Another way to view our parameters is by printing info about the parameter set.

In [ ]:
print_info(cell_parameters)

Parameters that take single numerical values (e.g. real, integers, booleans) can be directly modified.

In [ ]:
cell_parameters["PositiveElectrode"]["Coating"]["Thickness"] = 8.2e-5

Some parameters are described as functions or arrays, since the parameter value depends on other variables. For instance, the Open Circuit Potentials of the Active Materials depend on the lithium stoichiometry and temperature. When we're unsure about the type or meaning of a parameter, we can print information on individual parameters as well.

In [ ]:
print_info("OpenCircuitPotential", view = "CellParameters")

The cycling protocol parameters and the settings (model settings, simulation settings, solver settings) can be loaded, viewed and altered in the same way as the cell parameters. Let's load a default CCCV cycling protocol — we'll go into the settings later today.

In [ ]:
cycling_protocol = load_cycling_protocol(; from_default_set = "cccv")
print_info(cycling_protocol)

## 2 - Run a simulation ([docs](https://battmoteam.github.io/BattMo.jl/dev/manuals/user_guide/public_api))

Let's run a simple P2D simulation. We start again from the Chen 2020 cell parameter set and a constant current discharge cycling protocol.

In [ ]:
cell_parameters = load_cell_parameters(; from_default_set = "chen_2020")
cycling_protocol = load_cycling_protocol(; from_default_set = "cc_discharge")

Next, we select the default Lithium-Ion Battery model. A model can be thought of as a mathematical implementation of the electrochemical and transport phenomena occurring in a real battery cell — a system of partial differential equations together with their parameters, constants and boundary conditions. The default setup below is a basic P2D model, without current collectors or SEI growth.

In [ ]:
model = LithiumIonBattery()

The `LithiumIonBattery` constructor validates the model settings in the background. If the model setup is valid, we can create a `Simulation` object by passing the model, cell parameters and cycling protocol. The `Simulation` object validates the parameters and settings too — each set is checked for being sensible and complete.

In [ ]:
sim = Simulation(model, cell_parameters, cycling_protocol)

When the `Simulation` object is valid we can solve it by passing it to `solve`. As Julia is a compiled language, the first time we run a simulation it will take some time to compile the functions and structs it encounters — a second run will be much faster.

In [ ]:
output = solve(sim)

We can use built-in functions for quick plotting. The dashboard gives a quick overview of important output variables — `"contour"` shows position and time in one plot, `"line"` gives an interactive line plot with a time slider.

In [ ]:
plot_dashboard(output; plot_type = "contour")

In [ ]:
plot_dashboard(output; plot_type = "line")

Close all plotting windows before moving on.

In [ ]:
GLMakie.closeall()

## 3 - Output ([docs](https://battmoteam.github.io/BattMo.jl/dev/tutorials/3_handle_outputs))

In BattMo.jl the output variables are divided into three categories:
- **time series**: all variables that depend on time.
- **states**: all the state variables, which can depend on time, axial position and radial position.
- **metrics**: the calculated cell metrics, dependent on the cycle index.

Let's simulate a couple of constant current constant voltage cycles to look into these.

In [ ]:
cell_parameters = load_cell_parameters(; from_default_set = "chen_2020")
cycling_protocol = load_cycling_protocol(; from_default_set = "cccv")

cycling_protocol["TotalNumberOfCycles"] = 10

model = LithiumIonBattery()
sim = Simulation(model, cell_parameters, cycling_protocol)

output = solve(sim)

plot_dashboard(output; plot_type = "simple")

Let's see which output variables are available.

In [ ]:
print_info(output)

We can see the variables are divided into the three categories described above. Let's retrieve some time series data: voltage, current and time.

In [ ]:
time_series = output.time_series

t = time_series["Time"]
E = time_series["Voltage"]
I = time_series["Current"]

Let's also retrieve some state variables.

In [ ]:
states = output.states

electrolyte_concentration = states["Electrolyte"]["Concentration"]
electrolyte_potential = states["Electrolyte"]["Potential"]

We can print more information on an individual variable, for example to look into the difference between the electrode particle concentration and the surface concentration.

In [ ]:
print_info("NegativeElectrodeActiveMaterialParticleConcentration")

In [ ]:
print_info("NegativeElectrodeActiveMaterialSurfaceConcentration")

We can also retrieve metrics from the output, like the discharge capacity and round trip efficiency per cycle.

In [ ]:
metrics = output.metrics

discharge_capacity = metrics["DischargeCapacity"]
round_trip_efficiency = metrics["RoundTripEfficiency"]
cycle_index = metrics["CycleIndex"]

Let's plot the discharge capacity and round trip efficiency against cycle index.

In [ ]:
f = Figure(size = (1000, 400))

ax = Axis(f[1, 1], title = "Round trip efficiency", xlabel = "Cycle number / -", ylabel = "Efficiency / %")
scatterlines!(ax, cycle_index, round_trip_efficiency; linewidth = 4)

ax = Axis(f[2, 1], title = "Discharge capacity", xlabel = "Cycle number / -", ylabel = "Capacity / Ah")
scatterlines!(ax, cycle_index, discharge_capacity; linewidth = 4)

f

In [ ]:
GLMakie.closeall()

## Assignment — find the cliff

So far you've only run the `cc_discharge` protocol at its default rate. Real cells don't perform equally well at every discharge rate — capacity holds up for a while, then drops off a "cliff" at high rates, and beyond some point the solver may not converge at all.

Your task: sweep the `DRate` of the `cc_discharge` protocol from low to high (e.g. 0.2, 0.5, 1, 2, 3, 4, 5, 6 C), record the discharge capacity at each rate, and plot capacity vs. `DRate`. Wrap the simulation in a `try`/`catch` — at high rates the solver may fail outright rather than return a sensible result, so push `0.0` in that case.

In [ ]:
cell_parameters = load_cell_parameters(; from_default_set = "chen_2020")
discharge_protocol = load_cycling_protocol(; from_default_set = "cc_discharge")
model = LithiumIonBattery()

d_rates = [0.2, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
capacities = Float64[]

for d_rate in d_rates
    discharge_protocol["DRate"] = d_rate

    # --- build and solve the simulation, then push the discharge capacity (or 0.0 on failure) to `capacities` ---


    # --------------------------------------------------------------------------------------------------------
end

Now plot capacity vs. `DRate` to see where the cliff is.

In [ ]:
f = Figure(size = (700, 400))
ax = Axis(f[1, 1], title = "Rate capability", xlabel = "D-rate / C", ylabel = "Discharge capacity / Ah")
scatterlines!(ax, d_rates, capacities; linewidth = 4)
f

In [ ]:
GLMakie.closeall()

At roughly what rate does the cliff start? Keep this in mind — this afternoon's case-solving brief has a requirement that's exactly this trade-off (keeping at least 90% of capacity at 2C compared to 0.5C).